<a href="https://colab.research.google.com/github/vidit-demog/qlearning-regional-exposure-routing/blob/main/penn_homi_net_obj_share.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#Libraries used
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
!pip install geopandas contextily
import contextily as cx
import geopandas as gpd
from shapely.geometry import Point, LineString
import random
import signal


# **Penn Based Network: Q learning framework**

In [ ]:
#Reading the data
risk_data = pd.read_csv("/content/drive/MyDrive/df2_rscore.csv")
path_data = pd.read_csv("/content/drive/My Drive/pa_county_adjacency_distances.csv")
coords = pd.read_csv("/content/drive/My Drive/pa_county_centroids.csv")

#Merging the coordinates into node table
risk_data = risk_data.merge(coords, on="County", how="left")
display(risk_data.head(),
path_data.head())


In [ ]:
raw_table = (
    risk_data.groupby("RScore")
    .agg(
        n_counties=("County", "count"),
        counties=("County", lambda x: list(x))
    )
    .reset_index()
    .sort_values("RScore")
)

print(raw_table.to_string(index=False))


max_score = risk_data["RScore"].max()

conditions = [
    risk_data["RScore"] == 0,
    risk_data["RScore"].between(1, 4),
    risk_data["RScore"].between(5, 8),
    risk_data["RScore"].between(9, 11),
    risk_data["RScore"].between(12, 18),
    risk_data["RScore"].between(19, 40),
    risk_data["RScore"] >= 41,
]

labels = [
    "0",
    "1-4",
    "5-8",
    "9-11",
    "12-18",
    "19-40",
    f"41-{max_score}"
]

tmp = risk_data.copy()
tmp["RScore_bin"] = np.select(conditions, labels, default="Unknown")

order = ["0", "1-4", "5-8", "9-11", "12-18", "19-40", f"41-{max_score}"]

table = (
    tmp.groupby("RScore_bin", observed=True)
    .agg(
        n_counties=("County", "count"),
        counties=("County", lambda x: list(x))
    )
    .reindex(order)
    .reset_index()
)

print(table.to_string(index=False))

In [ ]:
#Initialize graph
G = nx.Graph()

#Add nodes with risk score

for _, row in risk_data.iterrows():
    G.add_node(
        row["County"],
        GEOID=row["GEOID"],
        risk_score=row["RScore"],
        rate=row["Rate"]
    )

#Add edges with distance

for _, row in path_data.iterrows():
    c1, c2 = row["county1"], row["county2"]

    # Add edge only if both counties exist in risk_data
    if G.has_node(c1) and G.has_node(c2):
        G.add_edge(
            c1,
            c2,
            distance_km=row["distance_km"]
        )

#Quick sanity checks

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

#inspect one node and one edge
example_node = list(G.nodes)[0]
example_edge = list(G.edges)[0]

print("Example node:", example_node, G.nodes[example_node])
print("Example edge:", example_edge, G.edges[example_edge])


In [ ]:
#Risk Distribution Map
G = nx.Graph()

for _, row in risk_data.iterrows():
    G.add_node(
        row["County"],
        risk_score=row["RScore"],
        rate=row["Rate"]
    )

for _, row in path_data.iterrows():
    G.add_edge(
        row["county1"],
        row["county2"],
        distance_km=row["distance_km"]
    )

#GeoDataFrame for mapping
gdf_nodes = gpd.GeoDataFrame(
    risk_data,
    geometry=gpd.points_from_xy(risk_data.lon, risk_data.lat),
    crs="EPSG:4326"  # lat/lon
)

#projecting to web mercator
gdf_nodes = gdf_nodes.to_crs(epsg=3857)

pos = {
    row["County"]: (row.geometry.x, row.geometry.y)
    for _, row in gdf_nodes.iterrows()
}

#colormap scaling
rates = [G.nodes[n]["risk_score"] for n in G.nodes() if G.nodes[n]["risk_score"] != -999.0]
vmin, vmax = min(rates), max(rates)

#Plot
fig, ax = plt.subplots(figsize=(12, 8))

nx.draw_networkx_edges(
    G, pos, ax=ax,
    width=[G.edges[e]["distance_km"] * 0.05 for e in G.edges()],
    alpha=0.5
)

nodes = nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=[G.nodes[n]["risk_score"] if G.nodes[n]["risk_score"] != -999.0 else vmin for n in G.nodes()],
    cmap=plt.cm.autumn_r,
    vmin=vmin,
    vmax=vmax
)

fig.colorbar(nodes, ax=ax, label="Risk Score")

# Adding real map
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.set_axis_off()
plt.savefig("/content/initial_graph_plot.png", bbox_inches='tight') # Save the plot
plt.show()

In [ ]:
# Deterministic Paths and Plots

import signal

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException


class RiskGraphEnv:

    def __init__(self, G, seed=42):
        self.graph = G
        self.distances = {(u,v): G.edges[u,v]["distance_km"] for u,v in G.edges()}
        self.risk = {n: G.nodes[n]["risk_score"] for n in G.nodes()}

        random.seed(seed)
        np.random.seed(seed)

        self.start = None
        self.goal = None
        self.current = None


    def set_start_goal(self, start, goal):

        self.start = start
        self.goal = goal
        self.current = start

        self.min_dist = nx.shortest_path_length(
            self.graph, start, goal, weight="distance_km"
        )

        def risk_weight(u,v,d):
            return self.risk[v]

        self.min_risk = nx.shortest_path_length(
            self.graph, start, goal, weight=risk_weight
        )

        self.min_dist = max(self.min_dist, 1e-8)
        self.min_risk = max(self.min_risk, 1e-8)


    def reset(self):
        self.current = self.start
        return self.current


    def actions(self, node):
        return list(self.graph.neighbors(node))


    def step(self, action):

        next_node = action

        dist = self.distances.get(
            (self.current, next_node),
            self.distances.get((next_node, self.current), 0)
        )

        r_dist = dist / self.min_dist
        r_risk = self.risk[next_node] / self.min_risk

        reward_vec = np.array([-r_dist, -r_risk])

        self.current = next_node
        done = (self.current == self.goal)

        if done:
            reward_vec += np.array([1.0, 0.0])

        return self.current, reward_vec, done


    def path_metrics(self, path):

        if path is None:
            return np.nan, np.nan

        length = sum(
            self.distances.get((path[i], path[i+1]),
                               self.distances.get((path[i+1], path[i]), 0))
            for i in range(len(path)-1)
        )

        risk = sum(self.risk[n] for n in path[1:])

        return length, risk



def q_learning_paths(env, weights_list,
                     episodes=8000,
                     alpha=0.1,
                     gamma=0.99,
                     epsilon=0.2,
                     max_steps=80):

    def learn_policy(w):

        Q = {s:{a:0 for a in env.actions(s)}
             for s in env.graph if env.actions(s)}

        for _ in range(episodes):

            s = env.reset()
            done = False

            while not done:

                if random.random() < epsilon:
                    a = random.choice(list(Q[s]))
                else:
                    # deterministic tie-breaking used for reproducibility of policy extraction
                    a = max(Q[s], key=lambda act: (Q[s][act], str(act)))

                s_next, r_vec, done = env.step(a)

                r = np.dot(w, r_vec)

                best_next = 0 if done else max(Q.get(s_next,{0:0}).values())

                Q[s][a] += alpha*(r + gamma*best_next - Q[s][a])

                s = s_next

        return Q


    def extract_path(Q):

        current = env.start
        visited=set()
        path=[current]

        for _ in range(max_steps):

            if current == env.goal:
                return path, "OPTIMAL"

            if current in visited:
                break

            visited.add(current)

            # deterministic extraction step (stable greedy policy)
            current = max(Q[current], key=lambda a: (Q[current][a], str(a)))
            path.append(current)

        return None, "NO PATH"


    results = []
    unique_paths = {}

    for i, w in enumerate(weights_list):

        # Ensuring reproducible exploration for policy learning
        random.seed(42 + i)
        np.random.seed(42 + i)

        try:

            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(100)

            Q = learn_policy(w)
            path,status = extract_path(Q)

            signal.alarm(0)

            length, risk = env.path_metrics(path)

            norm_len  = length / env.min_dist if path else np.nan
            norm_risk = risk   / env.min_risk if path else np.nan

            print(
                f"Weights {w} | {status} | "
                f"raw: dist={length:.2f}, risk={risk:.2f} | "
                f"norm: dist={norm_len:.3f}, risk={norm_risk:.3f}"
            )

            if path is not None:
                unique_paths.setdefault(tuple(path), []).append(w)

            results.append((w,status,length,risk,path))

        except TimeoutException:

            signal.alarm(0)

            print(f"Weights {w} | TIMEOUT after 100 seconds")

            results.append((w,"TIMEOUT",np.nan,np.nan,None))

    return results, unique_paths



def plot_paths_for_pair(G, gdf_nodes, unique_paths, title):

    fig, ax = plt.subplots(figsize=(10,8))

    pos = {
        row["County"]: (row.geometry.x, row.geometry.y)
        for _, row in gdf_nodes.iterrows()
    }

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.1, width=1)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=15)

    colors = plt.cm.tab10(np.linspace(0,1,len(unique_paths)))

    for i,(path,weights) in enumerate(unique_paths.items()):

        coords = [pos[n] for n in path]
        line = LineString(coords)

        offset = line.parallel_offset(1500*i, "left")
        xs, ys = offset.xy

        ax.plot(xs, ys, linewidth=4, color=colors[i],
                label=f"path {i+1}")

    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

    ax.set_title(title)
    ax.axis("off")
    ax.legend()

    plt.savefig(f"/content/paths_plot_{title.replace(' ', '_').replace('→', 'to')}.png", bbox_inches='tight') # Save the generated plot
    plt.show()



weights_list = [np.array([0.1+i/10, 0.9-i/10]) for i in range(10)]

pairs = [
    ("Beaver","York"),
    ("Greene","Pike"),
    ("Erie","Philadelphia")
]

gdf_nodes = gpd.GeoDataFrame(
    risk_data,
    geometry=gpd.points_from_xy(risk_data.lon, risk_data.lat),
    crs="EPSG:4326"
).to_crs(epsg=3857)


for start, goal in pairs:

    print("\n====================")
    print(start, "→", goal)

    env = RiskGraphEnv(G)
    env.set_start_goal(start, goal)

    results, unique_paths = q_learning_paths(env, weights_list)

    # Pareto scatter plot
    if len(results) > 0:
        plt.figure(figsize=(9,7))

        valid_results = [(l, r, w, status) for w, status, l, r, path in results if path is not None]

        if len(valid_results) > 0:
            lengths = [item[0] for item in valid_results]
            risks = [item[1] for item in valid_results]

            min_l, max_l = min(lengths), max(lengths)
            min_r, max_r = min(risks), max(risks)

            dx = max_l - min_l
            dy = max_r - min_r

            jitter_x = 0.02 * dx if dx > 0 else 0.01
            jitter_y = 0.02 * dy if dy > 0 else 0.01
        else:
            jitter_x = 0.01
            jitter_y = 0.01


        for w,status,l,r,path in results:
            if path is not None:
                jl = l + (random.random() - 0.5) * jitter_x
                jr = r + (random.random() - 0.5) * jitter_y
                label_str = f"W: [{w[0]:.1f}, {w[1]:.1f}] ({status})"
                plt.scatter(jl, jr, label=label_str)

        plt.xlabel("Path Length (Distance in km)")
        plt.ylabel("Total Risk Score")
        plt.title(f"Pareto Front for {start} → {goal}")
        plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0.)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(f"/content/pareto_plot_{start.replace(' ', '_')}_to_{goal.replace(' ', '_')}.png", bbox_inches='tight')
        plt.show()

    plot_paths_for_pair(
        G,
        gdf_nodes,
        unique_paths,
        f"{start} → {goal}"
    )

In [ ]:
# Stochastic Paths and Plots

# Timeout exception
import signal

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException


class RiskGraphEnv:

    def __init__(self, G, seed=42):
        self.graph = G
        self.distances = {(u,v): G.edges[u,v]["distance_km"] for u,v in G.edges()}
        self.base_risk = {n: G.nodes[n]["risk_score"] for n in G.nodes()}

        random.seed(seed)
        np.random.seed(seed)

        self.start = None
        self.goal = None
        self.current = None


    def set_start_goal(self, start, goal):
        self.start = start
        self.goal = goal
        self.current = start

        self.min_dist = nx.shortest_path_length(
            self.graph, start, goal, weight="distance_km"
        )

        def risk_weight(u,v,d):
            return self.base_risk[v]

        self.min_risk = nx.shortest_path_length(
            self.graph, start, goal, weight=risk_weight
        )

        self.min_dist = max(self.min_dist, 1e-8)
        self.min_risk = max(self.min_risk, 1e-8)


    def reset(self):
        self.current = self.start
        return self.current


    def actions(self, node):
        return list(self.graph.neighbors(node))


    def sample_node_risk(self, node):

        base = self.base_risk[node]

        neighbors = list(self.graph.neighbors(node))
        if neighbors:
            neighbor_risks = [self.base_risk[n] for n in neighbors]
            avg_neighbor_risk = np.mean(neighbor_risks)
        else:
            avg_neighbor_risk = 0

        influence = avg_neighbor_risk / (base + 1e-6)
        influence = np.clip(influence, 0, 1)

        min_val = 0
        max_val = 2 * base
        mean_val = base + influence * base

        std = 0.3 * base
        sampled = np.random.normal(mean_val, std)
        sampled = np.clip(sampled, min_val, max_val)

        return sampled


    def step(self, action):

        next_node = action

        dist = self.distances.get(
            (self.current, next_node),
            self.distances.get((next_node, self.current), 0)
        )

        r_dist = dist / self.min_dist
        r_risk = self.sample_node_risk(next_node) / self.min_risk

        reward_vec = np.array([-r_dist, -r_risk])

        self.current = next_node
        done = (self.current == self.goal)

        if done:
            reward_vec += np.array([1.0, 0.0])

        return self.current, reward_vec, done


    def path_metrics(self, path):
        if path is None:
            return np.nan, np.nan

        length = sum(
            self.distances.get((path[i], path[i+1]),
                               self.distances.get((path[i+1], path[i]), 0))
            for i in range(len(path)-1)
        )

        risk = sum(self.base_risk[n] for n in path[1:])

        return length, risk



def q_learning_paths(env, weights_list,
                     episodes=8000,
                     alpha=0.1,
                     gamma=0.99,
                     epsilon=0.2,
                     max_steps=80):

    def learn_policy(w):

        Q = {s:{a:0 for a in env.actions(s)}
             for s in env.graph if env.actions(s)}

        for _ in range(episodes):

            s = env.reset()
            done = False

            while not done:

                if random.random() < epsilon:
                    a = random.choice(list(Q[s]))
                else:
                    a = max(Q[s], key=lambda act: (Q[s][act], str(act)))

                s_next, r_vec, done = env.step(a)

                r = np.dot(w, r_vec)

                best_next = 0 if done else max(Q.get(s_next,{0:0}).values())

                Q[s][a] += alpha*(r + gamma*best_next - Q[s][a])

                s = s_next

        return Q


    def extract_path(Q):

        current = env.start
        visited=set()
        path=[current]

        for _ in range(max_steps):

            if current == env.goal:
                return path, "OPTIMAL"

            if current in visited:
                break

            visited.add(current)

            current = max(Q[current], key=lambda a: (Q[current][a], str(a)))
            path.append(current)

        return None, "NO PATH"


    results = []
    unique_paths = {}

    for i, w in enumerate(weights_list):

        # Ensuring per-weight reproducibility
        random.seed(42 + i)
        np.random.seed(42 + i)

        try:
            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(100)

            Q = learn_policy(w)
            path, status = extract_path(Q)

            signal.alarm(0)

            length, risk = env.path_metrics(path)

            norm_len  = length / env.min_dist if path else np.nan
            norm_risk = risk   / env.min_risk if path else np.nan

            print(
                f"Weights {w} | {status} | "
                f"raw: dist={length:.2f}, risk={risk:.2f} | "
                f"norm: dist={norm_len:.3f}, risk={norm_risk:.3f}"
            )

            if path is not None:
                unique_paths.setdefault(tuple(path), []).append(w)

            results.append((w,status,length,risk,path))

        except TimeoutException:
            signal.alarm(0)
            print(f"Weights {w} | TIMEOUT after 100 seconds")
            results.append((w,"TIMEOUT",np.nan,np.nan,None))

    return results, unique_paths



def plot_paths_for_pair(G_plot, gdf_nodes, unique_paths, title):

    fig, ax = plt.subplots(figsize=(10,8))

    pos = {
        row["County"]: (row.geometry.x, row.geometry.y)
        for _, row in gdf_nodes.iterrows()
    }

    nx.draw_networkx_edges(G_plot, pos, ax=ax, alpha=0.1, width=1)
    nx.draw_networkx_nodes(G_plot, pos, ax=ax, node_size=15)

    colors = plt.cm.tab10(np.linspace(0,1,len(unique_paths)))

    for i,(path,weights) in enumerate(unique_paths.items()):

        coords = [pos[n] for n in path]
        line = LineString(coords)

        offset = line.parallel_offset(1500*i, "left")
        xs, ys = offset.xy

        ax.plot(xs, ys, linewidth=4, color=colors[i],
                label=f"path {i+1}")

    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

    ax.set_title(title)
    ax.axis("off")
    ax.legend()

    plt.savefig(f"/content/stochastic_paths_plot_{title.replace(' ', '_').replace('→', 'to')}.png", bbox_inches='tight') # Save the generated plot
    plt.show()



weights_list = [np.array([0.1+i/10, 0.9-i/10]) for i in range(10)]

pairs = [
    ("Beaver","York"),
    ("Greene","Pike"),
    ("Erie","Philadelphia")
]

gdf_nodes = gpd.GeoDataFrame(
    risk_data,
    geometry=gpd.points_from_xy(risk_data.lon, risk_data.lat),
    crs="EPSG:4326"
).to_crs(epsg=3857)


for start, goal in pairs:

    print("\n====================")
    print(start, "→", goal)

    env = RiskGraphEnv(G)
    env.set_start_goal(start, goal)

    results, unique_paths = q_learning_paths(env, weights_list)

    if len(results) > 0:
        plt.figure(figsize=(9,7))

        valid_results = [(l, r, w, status) for w, status, l, r, path in results if path is not None]

        if len(valid_results) > 0:
            lengths = [item[0] for item in valid_results]
            risks = [item[1] for item in valid_results]

            dx = max(lengths) - min(lengths)
            dy = max(risks) - min(risks)

            jitter_x = 0.02 * dx if dx > 0 else 0.01
            jitter_y = 0.02 * dy if dy > 0 else 0.01
        else:
            jitter_x = 0.01
            jitter_y = 0.01


        for w,status,l,r,path in results:
            if path is not None:
                jl = l + (random.random() - 0.5) * jitter_x
                jr = r + (random.random() - 0.5) * jitter_y
                label_str = f"W: [{w[0]:.1f}, {w[1]:.1f}] ({status})"
                plt.scatter(jl, jr, label=label_str)

        plt.xlabel("Path Length (Distance in km)")
        plt.ylabel("Total Risk Score")
        plt.title(f"Pareto Front for {start} → {goal}")
        plt.legend(bbox_to_anchor=(1.05,1), loc="upper left", borderaxespad=0.)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(f"/content/stochastic_pareto_plot_{start.replace(' ', '_')}_to_{goal.replace(' ', '_')}.png", bbox_inches='tight')
        plt.show()

    plot_paths_for_pair(
        G,
        gdf_nodes,
        unique_paths,
        f"{start} → {goal}"
    )

In [ ]:
# Legacy code for all-pairs shortest path calculation (kept for reference)
#Build graph
G = nx.Graph()

#Assuming columns: county1, county2, distance
for _, row in path_data.iterrows():
    G.add_edge(row["county1"], row["county2"], weight=row["distance_km"])

#Compute all-pairs shortest path distances
lengths = dict(nx.all_pairs_dijkstra_path_length(G, weight="weight"))

#Convert to dataframe
rows = []
for src, targets in lengths.items():
    for dst, dist in targets.items():
        if src != dst:  # optional: remove self distances
            rows.append((src, dst, dist))

dist_df = pd.DataFrame(rows, columns=["source", "target", "distance_km"])

#Sort descending
dist_df = dist_df.sort_values(by="distance_km", ascending=False)

print(dist_df.head(20))

# 1. Build graph

G = nx.Graph()

for _, row in path_data.iterrows():
    G.add_edge(row["county1"], row["county2"], weight=row["distance_km"])

# 2. All-pairs shortest paths

lengths = nx.all_pairs_dijkstra_path_length(G, weight="weight")

# 3. Convert to DataFrame (unique unordered pairs only)

rows = []
for src, targets in lengths:
    for dst, dist in targets.items():
        if src < dst:   # key trick: avoids duplicates (A,B) vs (B,A)
            rows.append((src, dst, dist))

dist_df = pd.DataFrame(rows, columns=["county_a", "county_b", "distance_km"])


# 4. Rank distances (descending)

dist_df["rank"] = dist_df["distance_km"].rank(
    method="min", ascending=False
)

#Sort for inspection
dist_df = dist_df.sort_values(by="distance_km", ascending=False).reset_index(drop=True)


# 5. Query specific pairs

pairs_of_interest = [
    ("Beaver", "York"),
    ("Greene", "Pike"),
    ("Erie", "Philadelphia")
]

pairs_df = pd.DataFrame(pairs_of_interest, columns=["c1", "c2"])

#Normalize order to match (county a & b)
pairs_df["county_a"] = pairs_df[["c1", "c2"]].min(axis=1)
pairs_df["county_b"] = pairs_df[["c1", "c2"]].max(axis=1)

result = pairs_df.merge(
    dist_df[["county_a", "county_b", "distance_km", "rank"]],
    on=["county_a", "county_b"],
    how="left"
)

# Displaying the final results table
result = result[["county_a", "county_b", "distance_km", "rank"]]

print(result)

            source        target  distance_km
1847          Pike        Greene     701.3426
2045        Greene          Pike     701.3426
4421          Erie      Delaware     681.7583
3167      Delaware          Erie     681.7583
1715         Wayne        Greene     678.1937
2044        Greene         Wayne     678.1937
1846          Pike        Beaver     672.1851
65          Beaver          Pike     672.1851
1845          Pike    Washington     669.3706
131     Washington          Pike     669.3706
3101  Philadelphia          Erie     665.3647
4420          Erie  Philadelphia     665.3647
4419          Erie         Bucks     650.6254
3299         Bucks          Erie     650.6254
3298         Bucks        Beaver     650.4477
64          Beaver         Bucks     650.4477
1844          Pike     Allegheny     648.0764
197      Allegheny          Pike     648.0764
329       Lawrence          Pike     646.6490
1843          Pike      Lawrence     646.6490
  county_a      county_b  distance

In [ ]:
# Diagnostic check to see if Dijkstra's and RL converge
# under the (1.0, 0.0) pure distance objective

# Rebuild G for PA-only data with 'distance_km' edge attribute
G = nx.Graph()

for _, row in risk_data.iterrows():
    G.add_node(
        row["County"],
        GEOID=row["GEOID"],
        risk_score=row["RScore"],
        rate=row["Rate"]
    )

for _, row in path_data.iterrows():
    c1, c2 = row["county1"], row["county2"]

    # Add edge only if both counties exist in risk_data
    if G.has_node(c1) and G.has_node(c2):
        G.add_edge(
            c1,
            c2,
            distance_km=row["distance_km"]
        )


class RiskGraphEnv:

    def __init__(self, G, seed=42):

        self.graph = G

        self.distances = {
            (u, v): G.edges[u, v]["distance_km"]
            for u, v in G.edges()
        }

        self.risk = {
            n: G.nodes[n]["risk_score"]
            for n in G.nodes()
        }

        random.seed(seed)
        np.random.seed(seed)

        self.start = None
        self.goal = None
        self.current = None


    def set_start_goal(self, start, goal):

        self.start = start
        self.goal = goal
        self.current = start

        self.min_dist = nx.shortest_path_length(
            self.graph,
            start,
            goal,
            weight="distance_km"
        )

        def risk_weight(u, v, d):
            return self.risk[v]

        self.min_risk = nx.shortest_path_length(
            self.graph,
            start,
            goal,
            weight=risk_weight
        )

        self.min_dist = max(self.min_dist, 1e-8)
        self.min_risk = max(self.min_risk, 1e-8)


    def reset(self):

        self.current = self.start
        return self.current


    def actions(self, node):

        return list(self.graph.neighbors(node))


    def step(self, action):

        next_node = action

        dist = self.distances.get(
            (self.current, next_node),
            self.distances.get((next_node, self.current), 0)
        )

        r_dist = dist / self.min_dist
        r_risk = self.risk[next_node] / self.min_risk

        reward_vec = np.array([-r_dist, -r_risk])

        self.current = next_node
        done = (self.current == self.goal)

        if done:
            reward_vec += np.array([1.0, 0.0])

        return self.current, reward_vec, done


    def path_metrics(self, path):

        length = sum(
            self.distances.get(
                (path[i], path[i+1]),
                self.distances.get((path[i+1], path[i]), 0)
            )
            for i in range(len(path)-1)
        )

        risk = sum(
            self.risk[n]
            for n in path[1:]
        )

        return length, risk


def q_learning_paths(env,
                     weights_list,
                     episodes=80000,
                     alpha=0.1,
                     gamma=0.99,
                     epsilon=0.05,
                     max_steps=80):

    def learn_policy(w):

        Q = {
            s: {a: 0 for a in env.actions(s)}
            for s in env.graph
            if env.actions(s)
        }

        for _ in range(episodes):

            s = env.reset()
            done = False

            while not done:

                if random.random() < epsilon:
                    a = random.choice(list(Q[s]))
                else:
                    a = max(
                        Q[s],
                        key=lambda act: (Q[s][act], str(act))
                    )

                s_next, r_vec, done = env.step(a)

                r = np.dot(w, r_vec)

                best_next = (
                    0 if done
                    else max(Q.get(s_next, {0:0}).values())
                )

                Q[s][a] += alpha * (
                    r + gamma * best_next - Q[s][a]
                )

                s = s_next

        return Q


    def extract_path(Q):

        current = env.start
        visited = set()
        path = [current]

        for _ in range(max_steps):

            if current == env.goal:
                return path

            if current in visited:
                return None

            visited.add(current)

            current = max(
                Q[current],
                key=lambda a: (Q[current][a], str(a))
            )

            path.append(current)

        return None


    Q = learn_policy(weights_list[0])

    path = extract_path(Q)

    return path


start = "Greene"
goal = "Pike"

weights_list = [np.array([1.0, 0.0])]

env = RiskGraphEnv(G)
env.set_start_goal(start, goal)

rl_path = q_learning_paths(
    env,
    weights_list
)

rl_dist, rl_risk = env.path_metrics(rl_path)

print("\nRL PATH")
print("Path:", rl_path)
print(f"Distance: {rl_dist:.2f} km")
print(f"Risk: {rl_risk:.2f}")

dijkstra_path = nx.shortest_path(
    G,
    source=start,
    target=goal,
    weight="distance_km"
)

dijkstra_dist = nx.shortest_path_length(
    G,
    source=start,
    target=goal,
    weight="distance_km"
)

dijkstra_risk = sum(
    G.nodes[n]["risk_score"]
    for n in dijkstra_path[1:]
)

print("\nDIJKSTRA PATH")
print("Path:", dijkstra_path)
print(f"Distance: {dijkstra_dist:.2f} km")
print(f"Risk: {dijkstra_risk:.2f}")

print("\nCOMPARISON")

if rl_path == dijkstra_path:
    print("Exact path match between RL and Dijkstra.")
else:
    print("Paths differ.")

print(
    f"Distance difference: "
    f"{abs(rl_dist - dijkstra_dist):.2f} km"
)



RL PATH
Path: ['Greene', 'Fayette', 'Westmoreland', 'Cambria', 'Blair', 'Centre', 'Union', 'Northumberland', 'Schuylkill', 'Carbon', 'Monroe', 'Pike']
Distance: 717.66 km
Risk: 251.00

DIJKSTRA PATH
Path: ['Greene', 'Fayette', 'Westmoreland', 'Cambria', 'Blair', 'Huntingdon', 'Mifflin', 'Snyder', 'Northumberland', 'Schuylkill', 'Carbon', 'Monroe', 'Pike']
Distance: 701.34 km
Risk: 254.00

COMPARISON
Paths differ.
Distance difference: 16.32 km


In [ ]:
# Code for seeing the number of feasible rational paths generated for Beaver to York
start = "Beaver"
goal = "York"

# Spatial pruning
geo = gdf_nodes.set_index("County").geometry

start_pt = geo.loc[start]
goal_pt = geo.loc[goal]

buffer = 100000  # 100 km

minx = min(start_pt.x, goal_pt.x) - buffer
maxx = max(start_pt.x, goal_pt.x) + buffer
miny = min(start_pt.y, goal_pt.y) - buffer
maxy = max(start_pt.y, goal_pt.y) + buffer

allowed_nodes = [
    n for n in G.nodes()
    if (minx <= geo.loc[n].x <= maxx)
    and (miny <= geo.loc[n].y <= maxy)
]

G_sub = G.subgraph(allowed_nodes).copy()

print(f"Nodes before pruning: {G.number_of_nodes()}")
print(f"Nodes after pruning: {G_sub.number_of_nodes()}")

# Dijkstra baseline
dijkstra_dist = nx.shortest_path_length(
    G_sub,
    source=start,
    target=goal,
    weight="distance_km"
)

threshold = 1.5 * dijkstra_dist

print(f"Dijkstra distance: {dijkstra_dist:.2f} km")
print(f"Distance threshold: {threshold:.2f} km")

# DFS with pruning
valid_paths = []

def dfs(node, path, dist_so_far, visited):

    if dist_so_far > threshold:
        return

    if node == goal:
        valid_paths.append((list(path), dist_so_far))
        return

    for nbr in G_sub.neighbors(node):

        if nbr in visited:
            continue

        edge_dist = G_sub[node][nbr]["distance_km"]

        visited.add(nbr)
        path.append(nbr)

        dfs(
            nbr,
            path,
            dist_so_far + edge_dist,
            visited
        )

        path.pop()
        visited.remove(nbr)

dfs(
    start,
    [start],
    0.0,
    {start}
)

print(f"\nFeasible simple paths: {len(valid_paths)}")

for i, (p, d) in enumerate(valid_paths[:10], start=1):
    print(f"\nPath {i}")
    print(" -> ".join(p))
    print(f"Distance: {d:.2f} km")

# **Penn + NJ (Code for Appendix sections)**

In [ ]:
# Reading the data (PA+NJ)


risk_data2 = pd.read_csv("/content/drive/MyDrive/df_pa_nj_rscore.csv")
path_data2 = pd.read_csv("/content/drive/My Drive/pa_nj_county_adjacency_distances.csv")
coords2 = pd.read_csv("/content/drive/My Drive/pa_nj_county_centroids.csv")

# ensure consistent key in all tables
risk_data2["state_fips"] = risk_data2["GEOID"].astype(str).str[:2]
risk_data2["county_fips_padded"] = risk_data2["GEOID"].astype(str).str[2:].str.zfill(3)
risk_data2["county_id"] = risk_data2["state_fips"] + "_" + risk_data2["county_fips_padded"]

# coords already has county_id from pipeline
coords2 = coords2.copy()

# safety check: enforce correct join structure
if "county_id" not in coords2.columns:
    raise ValueError("Centroids file missing county_id — regenerate pipeline output")

# merge coordinates
risk_data2 = risk_data2.merge(
    coords2[["county_id", "lon", "lat"]],
    on="county_id",
    how="left"
)

missing = risk_data2["lon"].isna().sum()
print("Missing coordinate matches:", missing)



In [ ]:
raw_table2 = (
    risk_data2.groupby("RScore")
    .agg(
        n_counties=("County", "count"),
        counties=("County", lambda x: list(x))
    )
    .reset_index()
    .sort_values("RScore")
)

print(raw_table2.to_string(index=False))


max_score = risk_data2["RScore"].max()

conditions = [
    risk_data2["RScore"] == 0,
    risk_data2["RScore"].between(1, 4),
    risk_data2["RScore"].between(5, 8),
    risk_data2["RScore"].between(9, 11),
    risk_data2["RScore"].between(12, 18),
    risk_data2["RScore"].between(19, 40),
    risk_data2["RScore"] >= 41,
]

labels = [
    "0",
    "1-4",
    "5-8",
    "9-11",
    "12-18",
    "19-40",
    f"41-{max_score}"
]

tmp = risk_data2.copy()
tmp["RScore_bin"] = np.select(conditions, labels, default="Unknown")

order = ["0", "1-4", "5-8", "9-11", "12-18", "19-40", f"41-{max_score}"]

table = (
    tmp.groupby("RScore_bin", observed=True)
    .agg(
        n_counties=("County", "count"),
        counties=("County", lambda x: list(x))
    )
    .reindex(order)
    .reset_index()
)

print(table.to_string(index=False))

In [ ]:
# RISK DISTRIBUTION + GRAPH VISUALIZATION

# 1. Prepare risk_data2: Drop duplicates and create consistent county_id
risk_data2_cleaned = risk_data2.drop_duplicates(subset=['GEOID']).copy()

risk_data2_cleaned['state_fips'] = risk_data2_cleaned['GEOID'].astype(str).str[:2]
risk_data2_cleaned['county_fips_padded'] = risk_data2_cleaned['GEOID'].astype(str).str[2:].str.zfill(3)
risk_data2_cleaned['county_id'] = (
    risk_data2_cleaned['state_fips'] + '_' + risk_data2_cleaned['county_fips_padded']
)

# lookup maps
county_name_to_id = risk_data2_cleaned.set_index('County')['county_id'].to_dict()
geoid_int_to_id = risk_data2_cleaned.set_index('GEOID')['county_id'].to_dict()

# 2. Standardize path data
path_data2_processed = path_data2.copy()

def map_county_identifier(identifier):
    if isinstance(identifier, str) and '_' in identifier and len(identifier) == 6:
        return identifier

    try:
        geoid_int = int(identifier)
        if geoid_int in geoid_int_to_id:
            return geoid_int_to_id[geoid_int]
    except:
        pass

    return county_name_to_id.get(identifier, None)

path_data2_processed['county1_id'] = path_data2_processed['county1'].apply(map_county_identifier)
path_data2_processed['county2_id'] = path_data2_processed['county2'].apply(map_county_identifier)

path_data2_processed.dropna(subset=['county1_id', 'county2_id'], inplace=True)

# 3. Build graph
G = nx.Graph()

for _, row in risk_data2_cleaned.iterrows():
    G.add_node(
        row["county_id"],
        County_Name=row["County"],
        GEOID=row["GEOID"],
        risk_score=row["RScore"],
        rate=row["Rate"],
        lon=row["lon"],
        lat=row["lat"]
    )

for _, row in path_data2_processed.iterrows():
    if G.has_node(row["county1_id"]) and G.has_node(row["county2_id"]):
        G.add_edge(
            row["county1_id"],
            row["county2_id"],
            distance_km=row["distance_km"]
        )

# 4. GEOMETRY HANDLING

# build GeoDataFrame
gdf_nodes = gpd.GeoDataFrame(
    risk_data2_cleaned,
    geometry=gpd.points_from_xy(risk_data2_cleaned.lon, risk_data2_cleaned.lat),
    crs="EPSG:4326"
)

gdf_nodes = gdf_nodes.set_index('county_id')

# project ONCE properly
gdf_nodes_3857 = gdf_nodes.to_crs(epsg=3857)

# use representative point (more stable than centroid)
gdf_nodes_3857["rep_point"] = gdf_nodes_3857.geometry.representative_point()

pos = {
    idx: (row.rep_point.x, row.rep_point.y)
    for idx, row in gdf_nodes_3857.iterrows()
}

# 5. color scaling
rates = [
    G.nodes[n]["risk_score"]
    for n in G.nodes()
    if G.nodes[n]["risk_score"] != -999.0
]

vmin, vmax = min(rates), max(rates)

# 6. plot
fig, ax = plt.subplots(figsize=(12, 8))

distances = np.array([G.edges[e]["distance_km"] for e in G.edges()])
min_d, max_d = distances.min(), distances.max()

widths = [
    0.5 + 4.5 * (G.edges[e]["distance_km"] - min_d) / (max_d - min_d + 1e-9)
    for e in G.edges()
]

nx.draw_networkx_edges(G, pos, ax=ax, width=widths, alpha=0.5)

nx.draw_networkx_nodes(
    G,
    pos,
    ax=ax,
    node_color=[
        vmin if G.nodes[n]["risk_score"] == -999.0 else G.nodes[n]["risk_score"]
        for n in G.nodes()
    ],
    cmap=plt.cm.autumn_r,
    vmin=vmin,
    vmax=vmax
)

plt.colorbar(
    plt.cm.ScalarMappable(cmap=plt.cm.autumn_r),
    ax=ax,
    label="Risk Score"
)

cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.set_axis_off()
plt.savefig("/content/X_initial_graph_plot.png", bbox_inches='tight')
plt.show()

# 7. suspicious edge detection

bad_edges = []

for u, v in G.edges():
    x1, y1 = pos[u]
    x2, y2 = pos[v]

    dist_m = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

    # convert to km for interpretability
    dist_km = dist_m / 1000

    # flag absurd graph inconsistencies only
    if dist_km > 300:  # stricter + realistic threshold
        bad_edges.append((u, v, dist_km))

print(f"Suspicious edges: {len(bad_edges)}")

if len(bad_edges) > 0:
    print("Example:", bad_edges[:3])




In [ ]:
bad_edges = []

for u, v in G.edges():
    x1, y1 = pos[u]
    x2, y2 = pos[v]

    dist_m = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)
    dist_km = dist_m / 1000

    if dist_km > 300:
        bad_edges.append({
            "county1": u,
            "county2": v,
            "dist_km": dist_km,
            "road_km": G.edges[u, v]["distance_km"]
        })

print(f"Suspicious edges: {len(bad_edges)}\n")

for e in bad_edges:
    print(e)


for e in bad_edges:
    u, v = e["county1"], e["county2"]

    print(
        u, G.nodes[u]["County_Name"],
        " <-> ",
        v, G.nodes[v]["County_Name"]
    )




Suspicious edges: 0



In [ ]:
# Deterministic Paths & Plots For PA + NJ

import signal

# Timeout exception
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException


# build name → county_id lookup
name_to_id = risk_data2.drop_duplicates("County").set_index("County")["county_id"].to_dict()

# optional: for readable output
id_to_name = risk_data2.set_index("county_id")["County"].to_dict()


class RiskGraphEnv:

    def __init__(self, G, seed=42):
        self.graph = G
        self.distances = {(u,v): G.edges[u,v]["distance_km"] for u,v in G.edges()}
        self.risk = {n: G.nodes[n]["risk_score"] for n in G.nodes()}

        random.seed(seed)
        np.random.seed(seed)

        self.start = None
        self.goal = None
        self.current = None


    def set_start_goal(self, start, goal):

        self.start = start
        self.goal = goal
        self.current = start

        self.min_dist = nx.shortest_path_length(
            self.graph, start, goal, weight="distance_km"
        )

        def risk_weight(u,v,d):
            return self.risk[v]

        self.min_risk = nx.shortest_path_length(
            self.graph, start, goal, weight=risk_weight
        )

        self.min_dist = max(self.min_dist, 1e-8)
        self.min_risk = max(self.min_risk, 1e-8)


    def reset(self):
        self.current = self.start
        return self.current


    def actions(self, node):
        return list(self.graph.neighbors(node))


    def step(self, action):

        next_node = action

        dist = self.distances.get(
            (self.current, next_node),
            self.distances.get((next_node, self.current), 0)
        )

        r_dist = dist / self.min_dist
        r_risk = self.risk[next_node] / self.min_risk

        reward_vec = np.array([-r_dist, -r_risk])

        self.current = next_node
        done = (self.current == self.goal)

        # symmetric terminal reward
        if done:
            reward_vec += np.array([1.0, 1.0])

        return self.current, reward_vec, done


    def path_metrics(self, path):

        if path is None:
            return np.nan, np.nan

        length = sum(
            self.distances.get((path[i], path[i+1]),
                               self.distances.get((path[i+1], path[i]), 0))
            for i in range(len(path)-1)
        )

        risk = sum(self.risk[n] for n in path[1:])

        return length, risk



def q_learning_paths(env, weights_list,
                     episodes=8000,
                     alpha=0.1,
                     gamma=0.99,
                     epsilon=0.2,
                     max_steps=80):

    def learn_policy(w):

        Q = {s:{a:0 for a in env.actions(s)}
             for s in env.graph if env.actions(s)}

        for _ in range(episodes):

            s = env.reset()
            done = False

            while not done:

                if random.random() < epsilon:
                    a = random.choice(list(Q[s]))
                else:
                    # deterministic tie-breaking for reproducibility
                    a = max(Q[s], key=lambda act: (Q[s][act], str(act)))

                s_next, r_vec, done = env.step(a)

                r = np.dot(w, r_vec)

                best_next = 0 if done else max(Q.get(s_next,{0:0}).values())

                Q[s][a] += alpha*(r + gamma*best_next - Q[s][a])

                s = s_next

        return Q


    def extract_path(Q):

        current = env.start
        visited=set()
        path=[current]

        for _ in range(max_steps):

            if current == env.goal:
                return path, "OPTIMAL"

            if current in visited:
                break

            visited.add(current)

            # deterministic extraction
            current = max(Q[current], key=lambda a: (Q[current][a], str(a)))
            path.append(current)

        return None, "NO PATH"


    results = []
    unique_paths = {}

    for i, w in enumerate(weights_list):

        # Ensuring reproducible exploration for policy learning
        random.seed(42 + i)
        np.random.seed(42 + i)

        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(60)

        try:

            Q = learn_policy(w)
            path,status = extract_path(Q)

        except TimeoutException:

            Q = None
            path = None
            status = "TIMEOUT"

            print(f"Weights {w} | TIMEOUT after 60s")

        finally:
            signal.alarm(0)

        length, risk = env.path_metrics(path)

        norm_len  = length / env.min_dist if path else np.nan
        norm_risk = risk   / env.min_risk if path else np.nan

        print(
            f"Weights {w} | {status} | "
            f"raw: dist={length:.2f}, risk={risk:.2f} | "
            f"norm: dist={norm_len:.3f}, risk={norm_risk:.3f}"
        )

        if path is not None:
            unique_paths.setdefault(tuple(path), []).append(w)

        results.append((w,status,length,risk,path))

    return results, unique_paths



def plot_paths_for_pair(G, gdf_nodes, unique_paths, title):

    fig, ax = plt.subplots(figsize=(10,8))

    pos = {
        idx: (row.geometry.x, row.geometry.y)
        for idx, row in gdf_nodes.iterrows()
    }

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.1, width=1)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=15)

    colors = plt.cm.tab10(np.linspace(0,1,len(unique_paths)))

    for i,(path,weights) in enumerate(unique_paths.items()):

        coords = [pos[n] for n in path]
        line = LineString(coords)

        offset = line.parallel_offset(1500*i, "left")

        # robust handling for MultiLineString
        if offset.geom_type == "MultiLineString":
            offset = list(offset.geoms)[0]

        xs, ys = offset.xy

        ax.plot(xs, ys, linewidth=4, color=colors[i],
                label=f"path {i+1}")

    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

    ax.set_title(title)
    ax.axis("off")
    ax.legend()

    plt.savefig(
        f"/content/paths_plot_{title.replace(' ', '_').replace('→', 'to')}.png",
        bbox_inches='tight'
    )

    plt.show()



weights_list = [np.array([0.1+i/10, 0.9-i/10]) for i in range(10)]

pairs = [
    {"start": "Erie", "goal": "Cape May", "via": None},
    {"start": "Erie", "goal": "Cape May", "via": "Philadelphia"},
    {"start": "Sussex", "goal": "Cape May", "via": None}
]

pairs_mapped = []

for p in pairs:

    s, g, v = p["start"], p["goal"], p["via"]

    sid = name_to_id.get(s)
    gid = name_to_id.get(g)
    vid = name_to_id.get(v) if v else None

    if sid is None or gid is None or (v and vid is None):
        print(f"Skipping pair: {s}, {g}, via={v}")
        continue

    pairs_mapped.append((sid, gid, vid))


gdf_nodes = gpd.GeoDataFrame(
    risk_data2,
    geometry=gpd.points_from_xy(risk_data2.lon, risk_data2.lat),
    crs="EPSG:4326"
).set_index("county_id").to_crs(epsg=3857)


for start, goal, via in pairs_mapped:

    print("\n====================")

    if via:
        print(id_to_name[start], "→", id_to_name[via], "→", id_to_name[goal])
    else:
        print(id_to_name[start], "→", id_to_name[goal])

    env = RiskGraphEnv(G)

    if via is None:

        env.set_start_goal(start, goal)
        results, unique_paths = q_learning_paths(env, weights_list)

    else:

        env.set_start_goal(start, via)
        results1, paths1 = q_learning_paths(env, weights_list)

        env.set_start_goal(via, goal)
        results2, paths2 = q_learning_paths(env, weights_list)

        unique_paths = {}

        for p1 in paths1:
            for p2 in paths2:

                combined = p1[:-1] + p2
                unique_paths.setdefault(tuple(combined), []).append("via")

        results = []

        for path in unique_paths:

            length, risk = env.path_metrics(list(path))
            results.append((None, "VIA", length, risk, list(path)))

    if len(results) > 0:

        plt.figure(figsize=(9,7))

        valid_results = [
            (l, r)
            for _, _, l, r, path in results
            if path is not None
        ]

        if len(valid_results) > 0:

            lengths = [x[0] for x in valid_results]
            risks = [x[1] for x in valid_results]

            dx = max(lengths) - min(lengths)
            dy = max(risks) - min(risks)

            jitter_x = 0.02 * dx if dx > 0 else 0.01
            jitter_y = 0.02 * dy if dy > 0 else 0.01

        else:

            jitter_x = jitter_y = 0.01

        for _,status,l,r,path in results:

            if path is not None:

                jl = l + (random.random() - 0.5) * jitter_x
                jr = r + (random.random() - 0.5) * jitter_y

                plt.scatter(jl, jr, label=status)

        plt.xlabel("Path Length (Distance in km)")
        plt.ylabel("Total Risk Score")

        if via:
            plt.title(
                f"Pareto Front for "
                f"{id_to_name[start]} → {id_to_name[via]} → {id_to_name[goal]}"
            )
        else:
            plt.title(
                f"Pareto Front for "
                f"{id_to_name[start]} → {id_to_name[goal]}"
            )

        plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")

        plt.grid(True, linestyle='--', alpha=0.6)

        plt.tight_layout()

        plt.savefig(
            f"/content/pareto_plot_{start}_to_{goal}.png",
            bbox_inches='tight'
        )

        plt.show()

    print("\nPaths (ordered):")

    for i, (path, weights) in enumerate(unique_paths.items(), 1):

        named_path = [id_to_name[n] for n in path]

        print(f"Path {i}: " + " -> ".join(named_path))

    plot_paths_for_pair(
        G,
        gdf_nodes,
        unique_paths,
        f"{id_to_name[start]} → {id_to_name[goal]}"
        if not via
        else f"{id_to_name[start]} → {id_to_name[via]} → {id_to_name[goal]}"
    )

In [ ]:
import signal

# Stochastic Paths & Plots For PA + NJ

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException


name_to_id = risk_data2.drop_duplicates("County").set_index("County")["county_id"].to_dict()
id_to_name = risk_data2.set_index("county_id")["County"].to_dict()


class RiskGraphEnv:

    def __init__(self, G, seed=42):
        self.graph = G
        self.distances = {(u,v): G.edges[u,v]["distance_km"] for u,v in G.edges()}
        self.base_risk = {n: G.nodes[n]["risk_score"] for n in G.nodes()}

        random.seed(seed)
        np.random.seed(seed)

        self.start = None
        self.goal = None
        self.current = None


    def set_start_goal(self, start, goal):

        self.start = start
        self.goal = goal
        self.current = start

        self.min_dist = nx.shortest_path_length(
            self.graph, start, goal, weight="distance_km"
        )

        def risk_weight(u,v,d):
            return self.base_risk[v]

        self.min_risk = nx.shortest_path_length(
            self.graph, start, goal, weight=risk_weight
        )

        self.min_dist = max(self.min_dist, 1e-8)
        self.min_risk = max(self.min_risk, 1e-8)


    def reset(self):
        self.current = self.start
        return self.current


    def actions(self, node):
        return list(self.graph.neighbors(node))


    def sample_node_risk(self, node):

        base = self.base_risk[node]

        neighbors = list(self.graph.neighbors(node))

        if neighbors:
            neighbor_risks = [self.base_risk[n] for n in neighbors]
            avg_neighbor_risk = np.mean(neighbor_risks)
        else:
            avg_neighbor_risk = 0

        influence = avg_neighbor_risk / (base + 1e-6)
        influence = np.clip(influence, 0, 1)

        min_val = 0
        max_val = 2 * base
        mean_val = base + influence * base

        std = 0.3 * base

        sampled = np.random.normal(mean_val, std)
        sampled = np.clip(sampled, min_val, max_val)

        return sampled


    def step(self, action):

        next_node = action

        dist = self.distances.get(
            (self.current, next_node),
            self.distances.get((next_node, self.current), 0)
        )

        r_dist = dist / self.min_dist
        r_risk = self.sample_node_risk(next_node) / self.min_risk

        reward_vec = np.array([-r_dist, -r_risk])

        self.current = next_node
        done = (self.current == self.goal)

        if done:
            reward_vec += np.array([1.0, 0.0])

        return self.current, reward_vec, done


    def path_metrics(self, path):

        if path is None:
            return np.nan, np.nan

        length = sum(
            self.distances.get((path[i], path[i+1]),
                               self.distances.get((path[i+1], path[i]), 0))
            for i in range(len(path)-1)
        )

        risk = sum(self.base_risk[n] for n in path[1:])

        return length, risk



def q_learning_paths(env, weights_list,
                     episodes=8000,
                     alpha=0.1,
                     gamma=0.99,
                     epsilon=0.2,
                     max_steps=80):

    def learn_policy(w):

        Q = {
            s:{a:0 for a in env.actions(s)}
            for s in env.graph if env.actions(s)
        }

        for _ in range(episodes):

            s = env.reset()
            done = False

            while not done:

                if random.random() < epsilon:
                    a = random.choice(list(Q[s]))
                else:
                    # deterministic tie-breaking for reproducibility
                    a = max(Q[s], key=lambda act: (Q[s][act], str(act)))

                s_next, r_vec, done = env.step(a)

                r = np.dot(w, r_vec)

                best_next = 0 if done else max(Q.get(s_next,{0:0}).values())

                Q[s][a] += alpha * (
                    r + gamma * best_next - Q[s][a]
                )

                s = s_next

        return Q


    def extract_path(Q):

        current = env.start
        visited = set()
        path = [current]

        for _ in range(max_steps):

            if current == env.goal:
                return path, "OPTIMAL"

            if current in visited:
                break

            visited.add(current)

            # deterministic extraction for reproducibility
            current = max(
                Q[current],
                key=lambda a: (Q[current][a], str(a))
            )

            path.append(current)

        return None, "NO PATH"


    results = []
    unique_paths = {}

    for i, w in enumerate(weights_list):

        # reproducible stochastic sampling per weight
        random.seed(42 + i)
        np.random.seed(42 + i)

        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(100)

        try:

            Q = learn_policy(w)
            path, status = extract_path(Q)

        except TimeoutException:

            path = None
            status = "TIMEOUT"

            print(f"Weights {w} | TIMEOUT after 100s")

        finally:

            signal.alarm(0)

        length, risk = env.path_metrics(path)

        norm_len  = length / env.min_dist if path else np.nan
        norm_risk = risk   / env.min_risk if path else np.nan

        print(
            f"Weights {w} | {status} | "
            f"raw: dist={length:.2f}, risk={risk:.2f} | "
            f"norm: dist={norm_len:.3f}, risk={norm_risk:.3f}"
        )

        if path is not None:
            unique_paths.setdefault(tuple(path), []).append(w)

        results.append((w,status,length,risk,path))

    return results, unique_paths



def plot_paths_for_pair(G, gdf_nodes, unique_paths, title):

    fig, ax = plt.subplots(figsize=(10,8))

    pos = {
        idx: (row.geometry.x, row.geometry.y)
        for idx, row in gdf_nodes.iterrows()
    }

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.1, width=1)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=15)

    colors = plt.cm.tab10(np.linspace(0,1,len(unique_paths)))

    for i,(path,weights) in enumerate(unique_paths.items()):

        coords = [pos[n] for n in path]
        line = LineString(coords)

        offset = line.parallel_offset(1500*i, "left")

        # robustness for MultiLineString cases
        if offset.geom_type == "MultiLineString":
            offset = list(offset.geoms)[0]

        xs, ys = offset.xy

        ax.plot(
            xs, ys,
            linewidth=4,
            color=colors[i],
            label=f"path {i+1}"
        )

    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

    ax.set_title(title)
    ax.axis("off")
    ax.legend()

    plt.savefig(
        f"/content/stochastic_paths_plot_{title.replace(' ', '_').replace('→', 'to')}.png",
        bbox_inches='tight'
    )

    plt.show()



weights_list = [
    np.array([0.1+i/10, 0.9-i/10])
    for i in range(10)
]

pairs = [
    {"start": "Erie", "goal": "Cape May", "via": None},
    {"start": "Erie", "goal": "Cape May", "via": "Philadelphia"},
    {"start": "Sussex", "goal": "Cape May", "via": None}
]


pairs_mapped = []

for p in pairs:

    s, g, v = p["start"], p["goal"], p["via"]

    sid = name_to_id.get(s)
    gid = name_to_id.get(g)
    vid = name_to_id.get(v) if v else None

    if sid is None or gid is None or (v and vid is None):
        continue

    pairs_mapped.append((sid, gid, vid))


gdf_nodes = gpd.GeoDataFrame(
    risk_data2,
    geometry=gpd.points_from_xy(risk_data2.lon, risk_data2.lat),
    crs="EPSG:4326"
).set_index("county_id").to_crs(epsg=3857)


for start, goal, via in pairs_mapped:

    print("\n====================")

    if via:
        print(id_to_name[start], "→", id_to_name[via], "→", id_to_name[goal])
    else:
        print(id_to_name[start], "→", id_to_name[goal])

    env = RiskGraphEnv(G)

    if via is None:

        env.set_start_goal(start, goal)

        results, unique_paths = q_learning_paths(
            env,
            weights_list
        )

    else:

        env.set_start_goal(start, via)
        _, paths1 = q_learning_paths(env, weights_list)

        env.set_start_goal(via, goal)
        _, paths2 = q_learning_paths(env, weights_list)

        unique_paths = {}

        for p1 in paths1:
            for p2 in paths2:

                combined = p1[:-1] + p2

                unique_paths.setdefault(
                    tuple(combined),
                    []
                ).append("via")

        results = []

        for path in unique_paths:

            length, risk = env.path_metrics(list(path))

            results.append(
                (
                    np.array([0.5, 0.5]),
                    "VIA",
                    length,
                    risk,
                    list(path)
                )
            )

    # Pareto scatter plot
    if len(results) > 0:

        plt.figure(figsize=(9,7))

        valid_results = [
            (l, r, w, status)
            for w, status, l, r, path in results
            if path is not None
        ]

        if len(valid_results) > 0:

            lengths = [item[0] for item in valid_results]
            risks = [item[1] for item in valid_results]

            min_l, max_l = min(lengths), max(lengths)
            min_r, max_r = min(risks), max(risks)

            dx = max_l - min_l
            dy = max_r - min_r

            jitter_x = 0.02 * dx if dx > 0 else 0.01
            jitter_y = 0.02 * dy if dy > 0 else 0.01

        else:

            jitter_x = 0.01
            jitter_y = 0.01


        for w,status,l,r,path in results:

            if path is not None:

                jl = l + (random.random() - 0.5) * jitter_x
                jr = r + (random.random() - 0.5) * jitter_y

                if status != "VIA":
                    label_str = f"W: [{w[0]:.1f}, {w[1]:.1f}] ({status})"
                else:
                    label_str = status

                plt.scatter(jl, jr, label=label_str)

        plt.xlabel("Path Length (Distance in km)")
        plt.ylabel("Total Risk Score")

        if via:
            plt.title(
                f"Pareto Front for "
                f"{id_to_name[start]} → "
                f"{id_to_name[via]} → "
                f"{id_to_name[goal]}"
            )
        else:
            plt.title(
                f"Pareto Front for "
                f"{id_to_name[start]} → "
                f"{id_to_name[goal]}"
            )

        plt.legend(
            bbox_to_anchor=(1.05,1),
            loc="upper left",
            borderaxespad=0.
        )

        plt.grid(True, linestyle='--', alpha=0.6)

        plt.tight_layout()

        plt.savefig(
            f"/content/stochastic_pareto_plot_{start}_to_{goal}.png",
            bbox_inches='tight'
        )

        plt.show()


    print("\nPaths (ordered):")

    for i, (path, _) in enumerate(unique_paths.items(), 1):

        named_path = [id_to_name[n] for n in path]

        print(
            f"Path {i}: " +
            " -> ".join(named_path)
        )


    plot_paths_for_pair(
        G,
        gdf_nodes,
        unique_paths,
        f"{id_to_name[start]} → {id_to_name[goal]}"
        if not via
        else f"{id_to_name[start]} → {id_to_name[via]} → {id_to_name[goal]}"
    )

In [ ]:
print("\n==================== SELECTIVE DIJKSTRA (PA + NJ) ====================")

# 1. Build graph
G = nx.Graph()

for _, row in path_data2.iterrows():
    G.add_edge(row["county1"], row["county2"], weight=row["distance_km"])


# 2. Define only PA+NJ pairs (same structure as RL)
pairs = [
    {"start": "Erie", "goal": "Cape May", "via": None},
    {"start": "Erie", "goal": "Cape May", "via": "Philadelphia"},
    {"start": "Sussex", "goal": "Cape May", "via": None}
]

# Ensures name_to_id dictionary is available from prior cell executions.
if 'name_to_id' not in globals():
    raise NameError("name_to_id dictionary not found. Please run previous relevant cells.")

rows = []

# 3. Compute ONLY these paths
for p in pairs:

    s_name, g_name, v_name = p["start"], p["goal"], p["via"]

    # The county names are mapped to their respective IDs
    s_id = name_to_id.get(s_name)
    g_id = name_to_id.get(g_name)
    v_id = name_to_id.get(v_name) if v_name else None

    # Pairs are skipped if any county_id is not found
    if s_id is None or g_id is None or (v_name and v_id is None):
        print(f"Skipping pair due to missing ID: {s_name} -> {g_name} via {v_name}")
        continue

    if v_id is None:

        try:
            path = nx.shortest_path(G, s_id, g_id, weight="weight")
            dist = nx.shortest_path_length(G, s_id, g_id, weight="weight")

            rows.append({
                "source": s_name,
                "target": g_name,
                "via": None,
                "distance_km": dist,
                "path": path
            })

            print(f"{s_name} → {g_name}")
            print(" -> ".join(path))
            print(f"Distance: {dist:.2f}\n")

        except nx.NetworkXNoPath:
            print(f"No path: {s_name} → {g_name}")


    else:

        try:
            path1 = nx.shortest_path(G, s_id, v_id, weight="weight")
            dist1 = nx.shortest_path_length(G, s_id, v_id, weight="weight")

            path2 = nx.shortest_path(G, v_id, g_id, weight="weight")
            dist2 = nx.shortest_path_length(G, v_id, g_id, weight="weight")

            full_path = path1[:-1] + path2
            total_dist = dist1 + dist2

            rows.append({
                "source": s_name,
                "target": g_name,
                "via": v_name,
                "distance_km": total_dist,
                "path": full_path
            })

            print(f"{s_name} → {v_name} → {g_name}")
            print(" -> ".join(full_path))
            print(f"Distance: {total_dist:.2f}\n")

        except nx.NetworkXNoPath:
            print(f"No path: {s_name} → {v_name} → {g_name}")


# 4. Convert to DataFrame
dist_df = pd.DataFrame(rows)

# 5. Rank
dist_df["rank"] = dist_df["distance_km"].rank(method="min", ascending=False)

# Sorting descending
dist_df = dist_df.sort_values(by="distance_km", ascending=False).reset_index(drop=True)


print("\n==================== FINAL TABLE ====================")
print(dist_df[["source", "target", "via", "distance_km", "rank"]])